### SCI-Crawler

In [ ]:
import requests
import os
from xml.etree import ElementTree as ET

# 您的API密钥
api_key = "d04cb16625759c9b2146831e70f9d640"

# 构造DOI列表
doi_list = [
    # "10.1016/j.jallcom.2022.166624",
    # "10.1016/j.jallcom.2022.168658",
    # "10.1016/j.jallcom.2023.168865",
    # "10.1016/j.jallcom.2023.171786",
    # "10.1016/j.msea.2017.03.069",
    # "10.1016/j.msea.2017.04.085",
    # "10.1016/j.msea.2017.05.114",
    # "10.1016/j.msea.2019.01.002",
    # "10.1016/j.msea.2019.02.073",
    # "10.1016/j.msea.2022.143915",
    # "10.1016/j.msea.2024.146434",
    # "10.1016/j.matchar.2024.113693",
    # "10.1016/j.msea.2018.12.017",
    # "10.1016/j.msea.2020.140598",
    # "10.1016/j.msea.2022.143915",
    "10.1016/j.jallcom.2013.03.216",
    "10.1016/j.jallcom.2019.152915",
    "10.1016/j.vacuum.2021.110156",
    "10.1016/j.jallcom.2012.11.026"
]

# 设置请求头
headers = {
    "X-ELS-APIKey": api_key
}

# 指定存储路径
storage_path = "/Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/untitled folder2"

# 确保存储路径存在，如果不存在则创建
if not os.path.exists(storage_path):
    os.makedirs(storage_path)

# 循环遍历DOI列表
for doi in doi_list:
    # 构造API请求URL
    url = f"https://api.elsevier.com/content/article/doi/{doi}?httpAccept=text/xml"
    
    # 发送GET请求
    response = requests.get(url, headers=headers)
    
    # 检查请求是否成功
    if response.status_code == 200:
        # 解析XML响应数据
        root = ET.fromstring(response.content)
        
        # 构造文件名，使用DOI作为文件名
        file_name = f"{doi.replace('/', '_')}.xml"
        file_path = os.path.join(storage_path, file_name)
        
        # 打开文件并写入数据
        with open(file_path, 'wb') as file:  # 使用'wb'模式写入二进制数据
            file.write(ET.tostring(root, encoding='utf-8'))
            print(f"文件已保存至: {file_path}")
    else:
        print(f"请求失败，DOI: {doi}, 状态码：{response.status_code}")

文件已保存至: /Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/untitled folder2/10.1016_j.jallcom.2013.03.216.xml
文件已保存至: /Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/untitled folder2/10.1016_j.jallcom.2019.152915.xml
文件已保存至: /Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/untitled folder2/10.1016_j.vacuum.2021.110156.xml
文件已保存至: /Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/untitled folder2/10.1016_j.jallcom.2012.11.026.xml


### AutoFigExtractor

In [ ]:
from __future__ import annotations

import csv
import re
from pathlib import Path
from typing import Dict, List, Optional

import requests
from lxml import etree

# =========================================================
# 配置
# =========================================================
INPUT_XML_DIR = Path("/Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/XML")
OUTPUT_IMG_DIR = Path("/Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/Images")

OUTPUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
1
# 如果你有 Elsevier API Key，就填这里；没有就留空
ELSEVIER_API_KEY = "f55c59afdc3969fec43ca8773e901a00"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "*/*",
}
if ELSEVIER_API_KEY:
    HEADERS["X-ELS-APIKey"] = ELSEVIER_API_KEY

NS = {
    "default": "http://www.elsevier.com/xml/svapi/article/dtd",
    "ce": "http://www.elsevier.com/xml/common/dtd",
}

REQUEST_TIMEOUT = 60


# =========================================================
# 工具函数
# =========================================================
def safe_filename(name: str) -> str:
    return re.sub(r"[^\w\-.]+", "_", name)


def guess_ext(url: str, mimetype: str = "") -> str:
    url_l = url.lower()
    mime_l = mimetype.lower()

    if ".png" in url_l or "png" in mime_l:
        return ".png"
    if ".gif" in url_l or "gif" in mime_l:
        return ".gif"
    if ".tif" in url_l or ".tiff" in url_l or "tiff" in mime_l:
        return ".tif"
    return ".jpg"


def download_file(url: str, save_path: Path) -> tuple[bool, str]:
    try:
        r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()

        content_type = r.headers.get("Content-Type", "").lower()
        if "image" not in content_type and save_path.suffix.lower() not in {
            ".jpg", ".jpeg", ".png", ".gif", ".tif", ".tiff"
        }:
            return False, f"not image response: {content_type}"

        save_path.write_bytes(r.content)
        return True, "ok"
    except Exception as e:
        return False, str(e)


# =========================================================
# 解析 Elsevier XML
# =========================================================
def extract_image_links_from_elsevier_xml(xml_path: Path) -> List[Dict]:
    """
    从 Elsevier XML 中提取图片链接。
    只抓 <object ref="gr*|fx*"> 中的图像链接。
    同一个 ref 按 high > standard > thumbnail 选最优。
    """
    try:
        tree = etree.parse(str(xml_path))
    except Exception as e:
        print(f"[WARN] XML 解析失败: {xml_path.name} -> {e}")
        return []

    root = tree.getroot()
    objs = []

    for obj in root.findall(".//default:object", namespaces=NS):
        ref = obj.get("ref", "") or ""
        category = obj.get("category", "") or ""
        mimetype = obj.get("mimetype", "") or ""
        url = "".join(obj.itertext()).strip()

        # 只抓图像对象
        if not ref.startswith(("gr", "fx")):
            continue
        if not url.startswith("http"):
            continue

        # 尽量确认是图像
        if "image" not in mimetype.lower() and not re.search(
            r"\.(jpg|jpeg|png|gif|tif|tiff)(\?|$)", url, re.I
        ):
            continue

        objs.append({
            "ref": ref,
            "category": category,
            "mimetype": mimetype,
            "url": url,
        })

    # 同一个 ref 选最优类别
    priority = {"high": 3, "standard": 2, "thumbnail": 1}
    best: Dict[str, Dict] = {}

    for item in objs:
        ref = item["ref"]
        p = priority.get(item["category"], 0)
        if ref not in best or p > best[ref]["priority"]:
            best[ref] = {
                **item,
                "priority": p,
            }

    results = list(best.values())

    def ref_sort_key(x: Dict):
        m = re.findall(r"\d+", x["ref"])
        return int(m[0]) if m else 9999

    results.sort(key=ref_sort_key)
    return results


# =========================================================
# 主流程
# =========================================================
def run_batch(input_xml_dir: Path, output_img_dir: Path):
    xml_files = sorted(input_xml_dir.glob("*.xml"))
    if not xml_files:
        print(f"[WARN] 在 {input_xml_dir} 没找到 XML 文件")
        return

    log_rows = []
    total_links = 0
    total_ok = 0

    for xml_path in xml_files:
        print(f"\n{'=' * 60}")
        print(f"处理 XML: {xml_path.name}")

        image_infos = extract_image_links_from_elsevier_xml(xml_path)
        if not image_infos:
            print("  没找到图片链接")
            continue

        for item in image_infos:
            ref = item["ref"]
            category = item["category"]
            mimetype = item["mimetype"]
            url = item["url"]

            ext = guess_ext(url, mimetype)
            save_name = safe_filename(f"{xml_path.stem}_{ref}_{category}{ext}")
            save_path = output_img_dir / save_name

            total_links += 1

            if save_path.exists():
                print(f"  已存在，跳过: {save_name}")
                total_ok += 1
                log_rows.append([
                    xml_path.name, ref, category, url, str(save_path), "exists", "ok"
                ])
                continue

            ok, msg = download_file(url, save_path)
            if ok:
                total_ok += 1
                print(f"  ✓ 下载成功: {save_name}")
                status = "ok"
            else:
                print(f"  ✗ 下载失败: {save_name} | {msg}")
                status = "fail"

            log_rows.append([
                xml_path.name, ref, category, url, str(save_path), status, msg
            ])

    # 写日志
    log_csv = output_img_dir / "download_log.csv"
    with open(log_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "xml_file", "ref", "category", "url", "save_path", "status", "message"
        ])
        writer.writerows(log_rows)

    print("\n" + "=" * 60)
    print(f"图片链接总数: {total_links}")
    print(f"成功/已存在: {total_ok}")
    print(f"图片输出目录: {output_img_dir}")
    print(f"下载日志: {log_csv}")


if __name__ == "__main__":
    run_batch(INPUT_XML_DIR, OUTPUT_IMG_DIR)


处理 XML: 10.1016_j.jallcom.2012.11.026.xml
  已存在，跳过: 10.1016_j.jallcom.2012.11.026_gr1_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2012.11.026_gr2_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2012.11.026_gr3_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2012.11.026_gr4_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2012.11.026_gr5_high.jpg

处理 XML: 10.1016_j.jallcom.2013.03.216.xml
  已存在，跳过: 10.1016_j.jallcom.2013.03.216_gr1_standard.jpg
  已存在，跳过: 10.1016_j.jallcom.2013.03.216_gr2_standard.jpg
  已存在，跳过: 10.1016_j.jallcom.2013.03.216_gr3_standard.jpg

处理 XML: 10.1016_j.jallcom.2019.152915.xml
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr1_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr2_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr3_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr4_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr5_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr6_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr7_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr8_high.jpg
  已存在，跳过: 1

### Multi_Panel_By_Caption

In [ ]:
from __future__ import annotations

import csv
import re
import shutil
from pathlib import Path
from typing import Dict, List, Optional

import requests
from lxml import etree

# =========================================================
# 配置
# =========================================================
INPUT_XML_DIR = Path("/Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/XML")
OUTPUT_IMG_DIR = Path("/Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/Images")
MULTIPANEL_DIR = Path("/Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/MultiPanel_By_Caption")

OUTPUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
MULTIPANEL_DIR.mkdir(parents=True, exist_ok=True)

# 如果你有 Elsevier API Key，就填这里；没有就留空
ELSEVIER_API_KEY = "f55c59afdc3969fec43ca8773e901a00"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "*/*",
}
if ELSEVIER_API_KEY:
    HEADERS["X-ELS-APIKey"] = ELSEVIER_API_KEY

NS = {
    "default": "http://www.elsevier.com/xml/svapi/article/dtd",
    "ce": "http://www.elsevier.com/xml/common/dtd",
}

REQUEST_TIMEOUT = 60


# =========================================================
# 工具函数
# =========================================================
def safe_filename(name: str) -> str:
    return re.sub(r"[^\w\-.]+", "_", name)


def guess_ext(url: str, mimetype: str = "") -> str:
    url_l = url.lower()
    mime_l = mimetype.lower()

    if ".png" in url_l or "png" in mime_l:
        return ".png"
    if ".gif" in url_l or "gif" in mime_l:
        return ".gif"
    if ".tif" in url_l or ".tiff" in url_l or "tiff" in mime_l:
        return ".tif"
    return ".jpg"


def normalize_space(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def download_file(url: str, save_path: Path) -> tuple[bool, str]:
    try:
        r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()

        content_type = r.headers.get("Content-Type", "").lower()
        if "image" not in content_type and save_path.suffix.lower() not in {
            ".jpg", ".jpeg", ".png", ".gif", ".tif", ".tiff"
        }:
            return False, f"not image response: {content_type}"

        save_path.write_bytes(r.content)
        return True, "ok"
    except Exception as e:
        return False, str(e)


# =========================================================
# Caption 解析与多子图判断
# =========================================================
def clean_caption_text(text: str) -> str:
    text = normalize_space(text)
    # 常见 unicode 括号归一
    text = text.replace("（", "(").replace("）", ")")
    return text


def longest_consecutive_letter_run(text: str) -> List[str]:
    """
    从 caption 中找最长的连续字母序列。
    优先找 a,b,c,d... 这样的顺序，用于判断多子图。
    """
    text = clean_caption_text(text)

    patterns = [
        r"\(([A-Ia-i])\)",                 # (a) (b) (c)
        r"(?<![A-Za-z0-9])([A-Ia-i])(?=[\s,;:.])",  # a, b, c
        r"(?<![A-Za-z0-9])([A-Ia-i])(?![A-Za-z0-9])",  # 单独字母
    ]

    hits: List[tuple[int, str]] = []

    for pat in patterns:
        for m in re.finditer(pat, text):
            ch = m.group(1).lower()
            hits.append((m.start(), ch))

    if not hits:
        return []

    hits.sort(key=lambda x: x[0])

    # 去掉位置极近的重复字母
    filtered: List[tuple[int, str]] = []
    last_pos = -999
    last_ch = ""
    for pos, ch in hits:
        if pos - last_pos < 2 and ch == last_ch:
            continue
        filtered.append((pos, ch))
        last_pos = pos
        last_ch = ch

    if not filtered:
        return []

    best_run: List[str] = []
    cur_run: List[str] = [filtered[0][1]]

    for i in range(1, len(filtered)):
        prev = filtered[i - 1][1]
        cur = filtered[i][1]

        if ord(cur) == ord(prev) + 1:
            cur_run.append(cur)
        elif cur != prev:
            if len(cur_run) > len(best_run):
                best_run = cur_run[:]
            cur_run = [cur]

    if len(cur_run) > len(best_run):
        best_run = cur_run[:]

    return best_run


def detect_multi_panel_from_caption(caption: str) -> Dict:
    """
    根据 caption 中 a,b,c,d... 的连续顺序判断是否存在多子图。
    """
    caption = clean_caption_text(caption)
    run = longest_consecutive_letter_run(caption)

    panel_count = len(run)
    is_multi = panel_count >= 2

    if panel_count == 0:
        layout_guess = "single"
    elif panel_count == 2:
        layout_guess = "2-panel"
    elif panel_count == 4:
        layout_guess = "4-panel"
    elif panel_count == 9:
        layout_guess = "9-panel"
    else:
        layout_guess = f"{panel_count}-panel"

    return {
        "is_multi_panel": is_multi,
        "panel_count_by_caption": panel_count,
        "panel_letters": "".join(run),
        "layout_guess": layout_guess,
    }


# =========================================================
# Elsevier XML 解析
# =========================================================
def extract_caption_map_from_elsevier_xml(root) -> Dict[str, str]:
    """
    建立 ref -> caption 的映射
    优先从 <ce:figure> 中提取 caption。
    """
    caption_map: Dict[str, str] = {}

    figure_nodes = root.findall(".//ce:figure", namespaces=NS)
    for fig in figure_nodes:
        ref = fig.get("id", "") or fig.get("ref", "") or ""

        # caption 可能在 ce:caption 里
        caption_parts = []
        for cap in fig.findall(".//ce:caption", namespaces=NS):
            txt = "".join(cap.itertext()).strip()
            if txt:
                caption_parts.append(txt)

        # fallback: 有些 caption 在 para 里
        if not caption_parts:
            for para in fig.findall(".//ce:para", namespaces=NS):
                txt = "".join(para.itertext()).strip()
                if txt:
                    caption_parts.append(txt)

        caption = clean_caption_text(" ".join(caption_parts)) if caption_parts else ""

        if ref and caption:
            caption_map[ref] = caption

    return caption_map


def extract_label_map_from_elsevier_xml(root) -> Dict[str, str]:
    """
    建立 ref -> label 的映射，比如 Figure 1, Fig. 2
    """
    label_map: Dict[str, str] = {}

    figure_nodes = root.findall(".//ce:figure", namespaces=NS)
    for fig in figure_nodes:
        ref = fig.get("id", "") or fig.get("ref", "") or ""
        label_text = ""

        label_node = fig.find(".//ce:label", namespaces=NS)
        if label_node is not None:
            label_text = clean_caption_text("".join(label_node.itertext()))

        if ref and label_text:
            label_map[ref] = label_text

    return label_map


def match_ref_to_caption(ref: str, caption_map: Dict[str, str], label_map: Dict[str, str]) -> tuple[str, str]:
    """
    尝试把 object ref（如 gr1, fx1）匹配到 figure caption。
    先精确匹配，再按数字匹配。
    返回: (label, caption)
    """
    if ref in caption_map:
        return label_map.get(ref, ""), caption_map[ref]

    # 按数字后缀匹配，如 gr1 -> 某个 id 里含 1
    m = re.findall(r"\d+", ref)
    num = m[0] if m else None

    if num:
        for k, v in caption_map.items():
            nums = re.findall(r"\d+", k)
            if nums and nums[0] == num:
                return label_map.get(k, ""), v

    return "", ""


def extract_image_links_from_elsevier_xml(xml_path: Path) -> List[Dict]:
    """
    从 Elsevier XML 中提取图片链接，并尝试关联 caption。
    只抓 <object ref="gr*|fx*"> 中的图像链接。
    同一个 ref 按 high > standard > thumbnail 选最优。
    """
    try:
        tree = etree.parse(str(xml_path))
    except Exception as e:
        print(f"[WARN] XML 解析失败: {xml_path.name} -> {e}")
        return []

    root = tree.getroot()
    caption_map = extract_caption_map_from_elsevier_xml(root)
    label_map = extract_label_map_from_elsevier_xml(root)

    objs = []

    for obj in root.findall(".//default:object", namespaces=NS):
        ref = obj.get("ref", "") or ""
        category = obj.get("category", "") or ""
        mimetype = obj.get("mimetype", "") or ""
        url = "".join(obj.itertext()).strip()

        if not ref.startswith(("gr", "fx")):
            continue
        if not url.startswith("http"):
            continue

        if "image" not in mimetype.lower() and not re.search(
            r"\.(jpg|jpeg|png|gif|tif|tiff)(\?|$)", url, re.I
        ):
            continue

        fig_label, caption = match_ref_to_caption(ref, caption_map, label_map)

        objs.append({
            "ref": ref,
            "category": category,
            "mimetype": mimetype,
            "url": url,
            "figure_label": fig_label,
            "caption": caption,
        })

    priority = {"high": 3, "standard": 2, "thumbnail": 1}
    best: Dict[str, Dict] = {}

    for item in objs:
        ref = item["ref"]
        p = priority.get(item["category"], 0)
        if ref not in best or p > best[ref]["priority"]:
            best[ref] = {
                **item,
                "priority": p,
            }

    results = list(best.values())

    def ref_sort_key(x: Dict):
        m = re.findall(r"\d+", x["ref"])
        return int(m[0]) if m else 9999

    results.sort(key=ref_sort_key)
    return results


# =========================================================
# 主流程
# =========================================================
def run_batch(input_xml_dir: Path, output_img_dir: Path, multipanel_dir: Path):
    xml_files = sorted(input_xml_dir.glob("*.xml"))
    if not xml_files:
        print(f"[WARN] 在 {input_xml_dir} 没找到 XML 文件")
        return

    log_rows = []
    total_links = 0
    total_ok = 0
    total_multi = 0

    for xml_path in xml_files:
        print(f"\n{'=' * 60}")
        print(f"处理 XML: {xml_path.name}")

        image_infos = extract_image_links_from_elsevier_xml(xml_path)
        if not image_infos:
            print("  没找到图片链接")
            continue

        for item in image_infos:
            ref = item["ref"]
            category = item["category"]
            mimetype = item["mimetype"]
            url = item["url"]
            fig_label = item.get("figure_label", "")
            caption = item.get("caption", "")

            multi_info = detect_multi_panel_from_caption(caption)

            ext = guess_ext(url, mimetype)
            save_name = safe_filename(f"{xml_path.stem}_{ref}_{category}{ext}")
            save_path = output_img_dir / save_name

            total_links += 1

            if save_path.exists():
                print(f"  已存在，跳过: {save_name}")
                total_ok += 1
                status = "exists"
                msg = "ok"
            else:
                ok, msg = download_file(url, save_path)
                if ok:
                    total_ok += 1
                    print(f"  ✓ 下载成功: {save_name}")
                    status = "ok"
                else:
                    print(f"  ✗ 下载失败: {save_name} | {msg}")
                    status = "fail"

            # 如果 caption 判断为多子图，复制到另一个文件夹
            copied_multi = ""
            if save_path.exists() and multi_info["is_multi_panel"]:
                multi_save_path = multipanel_dir / save_name
                if not multi_save_path.exists():
                    shutil.copy2(save_path, multi_save_path)
                copied_multi = str(multi_save_path)
                total_multi += 1

            log_rows.append([
                xml_path.name,
                ref,
                category,
                fig_label,
                url,
                str(save_path),
                status,
                msg,
                caption,
                multi_info["is_multi_panel"],
                multi_info["panel_count_by_caption"],
                multi_info["panel_letters"],
                multi_info["layout_guess"],
                copied_multi,
            ])

    # 下载日志 + caption 日志
    log_csv = output_img_dir / "figure_caption_log.csv"
    with open(log_csv, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow([
            "xml_file",
            "ref",
            "category",
            "figure_label",
            "url",
            "save_path",
            "status",
            "message",
            "caption",
            "is_multi_panel_by_caption",
            "panel_count_by_caption",
            "panel_letters",
            "layout_guess",
            "copied_to_multi_panel_dir",
        ])
        writer.writerows(log_rows)

    print("\n" + "=" * 60)
    print(f"图片链接总数: {total_links}")
    print(f"成功/已存在: {total_ok}")
    print(f"caption 判断为多子图数量: {total_multi}")
    print(f"图片输出目录: {output_img_dir}")
    print(f"多子图图片目录: {multipanel_dir}")
    print(f"日志文件: {log_csv}")


if __name__ == "__main__":
    run_batch(INPUT_XML_DIR, OUTPUT_IMG_DIR, MULTIPANEL_DIR)


处理 XML: 10.1016_j.jallcom.2012.11.026.xml
  已存在，跳过: 10.1016_j.jallcom.2012.11.026_gr1_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2012.11.026_gr2_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2012.11.026_gr3_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2012.11.026_gr4_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2012.11.026_gr5_high.jpg

处理 XML: 10.1016_j.jallcom.2013.03.216.xml
  已存在，跳过: 10.1016_j.jallcom.2013.03.216_gr1_standard.jpg
  已存在，跳过: 10.1016_j.jallcom.2013.03.216_gr2_standard.jpg
  已存在，跳过: 10.1016_j.jallcom.2013.03.216_gr3_standard.jpg

处理 XML: 10.1016_j.jallcom.2019.152915.xml
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr1_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr2_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr3_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr4_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr5_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr6_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr7_high.jpg
  已存在，跳过: 10.1016_j.jallcom.2019.152915_gr8_high.jpg
  已存在，跳过: 1

### Classification

In [ ]:
from __future__ import annotations

import os
import re
import csv
import time
import base64
from typing import List, Dict, Any, Set

from tqdm import tqdm
from openai import OpenAI
import openai

# =======================
# Global config
# =======================
api_key = "sk-or-v1"   # Fill in your own OpenRouter / API key
base_url = "https://openrouter.ai/api/v1"
MODEL_NAME = "qwen/qwen3.5-9b"
# MODEL_NAME = "openai/gpt-5.4"
# MODEL_NAME = "deepseek/deepseek-v3.2"
# MODEL_NAME = "anthropic/claude-3.5-haiku"

input_data_path = "/Users/zixuanzhao/Desktop/MKG/AutoFigExtractor/Hardness2"
output_csv_path = "/Users/zixuanzhao/Desktop/MKG/AutoFigExtractor/image_types.csv"
failed_path = "/Users/zixuanzhao/Desktop/MKG/AutoFigExtractor/Failed2"

client = OpenAI(api_key=api_key, base_url=base_url)

os.makedirs(failed_path, exist_ok=True)

# =======================
# Prompt
# =======================
system_content = """
You are classifying a figure from a materials science paper.

Classes:
0 = microstructure (SEM, TEM, EBSD, OM, elemental maps)
1 = diffraction (XRD patterns, SAED, diffraction spots/rings)
2 = schematic (mechanism diagrams, workflows, process illustrations)
3 = curve (line plots, stress-strain curves, aging/hardening curves)
4 = statistical_plot (bar charts, scatter plots, histograms, box plots)
5 = other (unclear or none of the above)

Rules:
- XRD or diffraction spots → 1 (not 3)
- SEM/TEM/EBSD/OM or mapping → 0
- Continuous lines → 3
- Bars or discrete points → 4
- Diagrams with arrows/process → 2

Return only one number (0-5). No explanation.
""".strip()

system_instructions = {"role": "system", "content": system_content}

PROMPT_TEXT = "Classify this scientific image into exactly one class."


# =======================
# Utilities
# =======================
def detect_file_exist(file_path: str) -> bool:
    return os.path.exists(file_path)


def list_image_files(folder: str) -> List[str]:
    valid_ext = {".png", ".jpg", ".jpeg"}
    files = []
    for f in os.listdir(folder):
        if f == ".DS_Store":
            continue
        ext = os.path.splitext(f)[1].lower()
        if ext in valid_ext:
            files.append(f)
    return sorted(files)


def move_non_image_files(folder: str, failed_folder: str):
    valid_ext = {".png", ".jpg", ".jpeg"}
    for f in os.listdir(folder):
        if f == ".DS_Store":
            continue
        full_path = os.path.join(folder, f)
        if not os.path.isfile(full_path):
            continue
        ext = os.path.splitext(f)[1].lower()
        if ext not in valid_ext:
            dst = os.path.join(failed_folder, f)
            try:
                os.rename(full_path, dst)
                print(f"⏭️ Skip non-image file: {f}")
            except Exception as e:
                print(f"⚠️ Move failed: {f} | {e}")


def encode_image_base64(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def build_data_url(image_path: str) -> str:
    ext = os.path.splitext(image_path)[1].lower()
    mime = "image/png"
    if ext in [".jpg", ".jpeg"]:
        mime = "image/jpeg"
    b64 = encode_image_base64(image_path)
    return f"data:{mime};base64,{b64}"


def normalize_type(raw_text: str) -> str:
    """
    模型返回 0-5，转成类别名；也兼容偶发文字输出。
    """
    if not raw_text:
        return "other"

    text = raw_text.strip().lower()

    digit_to_type = {
        "0": "microstructure",
        "1": "diffraction",
        "2": "schematic",
        "3": "curve",
        "4": "statistical_plot",
        "5": "other",
    }

    if text in digit_to_type:
        return digit_to_type[text]

    allowed = [
        "microstructure",
        "diffraction",
        "schematic",
        "curve",
        "statistical_plot",
        "other",
    ]

    if text in allowed:
        return text

    for item in allowed:
        if item in text:
            return item

    if "bar" in text or "scatter" in text or "histogram" in text or "statistical" in text:
        return "statistical_plot"
    if "xrd" in text or "saed" in text or "diffraction" in text:
        return "diffraction"
    if "tem" in text or "sem" in text or "ebsd" in text or "microstructure" in text or "mapping" in text:
        return "microstructure"
    if "curve" in text or "line plot" in text or "stress-strain" in text:
        return "curve"
    if "schematic" in text or "diagram" in text or "illustration" in text or "workflow" in text:
        return "schematic"

    return "other"


def load_processed_images(csv_path: str) -> Set[str]:
    """
    读取已处理过的图片名，支持断点续跑。
    """
    processed = set()
    if not os.path.exists(csv_path):
        return processed

    try:
        with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                img_name = str(row.get("image_name", "")).strip()
                if img_name:
                    processed.add(img_name)
    except Exception as e:
        print(f"⚠️ Failed to load existing CSV: {e}")

    return processed


def init_csv_if_needed(csv_path: str):
    """
    如果 CSV 不存在，则写表头
    """
    if os.path.exists(csv_path):
        return

    fieldnames = ["image_name", "image_type", "raw_output"]
    with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()


def append_result_to_csv(row: Dict[str, Any], csv_path: str):
    """
    每处理完一张图立刻写一行，防止中断丢结果
    """
    fieldnames = ["image_name", "image_type", "raw_output"]
    with open(csv_path, "a", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(row)


# =======================
# API call
# =======================
def ask_model_with_image(image_path: str) -> str:
    data_url = build_data_url(image_path)

    messages = [
        system_instructions,
        {
            "role": "user",
            "content": [
                {"type": "text", "text": PROMPT_TEXT},
                {
                    "type": "image_url",
                    "image_url": {"url": data_url}
                }
            ]
        }
    ]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=0,
    )

    return response.choices[0].message.content.strip()


# =======================
# Core processing
# =======================
def process_one_file(input_file_path: str) -> Dict[str, Any]:
    file = os.path.basename(input_file_path)

    print(f"📤 Submitting: {file}")

    max_retries = 3
    retries = 0
    prediction = None

    while retries < max_retries:
        try:
            prediction = ask_model_with_image(input_file_path)
            break
        except openai.InternalServerError as e:
            retries += 1
            print(f"⚠️ ServerError: {file} | {e}. Retrying ({retries}/{max_retries})")
            time.sleep(2 ** retries)
        except Exception as e:
            print(f"❌ API call failed: {file} | {e}")
            return {
                "image_name": file,
                "image_type": "api_failed",
                "raw_output": str(e),
            }

    if prediction is None:
        print(f"❌ Max retries reached: {file}")
        return {
            "image_name": file,
            "image_type": "api_failed",
            "raw_output": "max retries reached",
        }

    image_type = normalize_type(prediction)

    print(f"✅ Done: {file} -> {image_type} | raw={prediction}")

    return {
        "image_name": file,
        "image_type": image_type,
        "raw_output": prediction,
    }


# =======================
# Main
# =======================
def main():
    os.makedirs(failed_path, exist_ok=True)

    # 先把 txt 等非图片移走
    move_non_image_files(input_data_path, failed_path)

    # 初始化 CSV
    init_csv_if_needed(output_csv_path)

    # 读取已处理记录，支持断点续跑
    processed_images = load_processed_images(output_csv_path)
    print(f"📌 Already processed: {len(processed_images)} images")

    # 只处理图片
    image_files = list_image_files(input_data_path)
    if not image_files:
        print("No image files found.")
        return

    # 跳过已处理
    pending_files = [f for f in image_files if f not in processed_images]

    print("Processing image folder:", input_data_path)
    print(f"🖼️ Total images: {len(image_files)}")
    print(f"⏭️ Skipped already processed: {len(image_files) - len(pending_files)}")
    print(f"🚀 Remaining to process: {len(pending_files)}")

    if not pending_files:
        print("All images already processed.")
        return

    try:
        for idx, file in enumerate(tqdm(pending_files), start=1):
            input_file_path = os.path.join(input_data_path, file)
            row = process_one_file(input_file_path)

            # 边运行边写出
            append_result_to_csv(row, output_csv_path)

            print(f"💾 Saved to CSV immediately: {file} ({idx}/{len(pending_files)})")

    except KeyboardInterrupt:
        print("\n⛔ Interrupted by user. Progress has already been saved to CSV.")
        return
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        print("Progress before the error has already been saved to CSV.")
        return

    print(f"\n🎉 Completed! CSV saved at: {output_csv_path}")


if __name__ == "__main__":
    main()

### Metrics

In [8]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# =========================
# 1. 读取数据
# =========================
file_path = "/Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/LLM/Results.xlsx"
df = pd.read_excel(file_path)

df.columns = df.columns.str.strip()
print("列名:", df.columns.tolist())

# =========================
# 2. 设置 GT 列
# =========================
GT_COL = "GT_output"

# =========================
# 3. 标签映射
# =========================
label_map = {
    "microstructure": 0,
    "diffraction": 1,
    "curve": 2,
    "statistical_plot": 3,
    "schematic": 4,
    "other": 5,

    "statistical": 3,
    "statisticalplot": 3,
    "plot": 3,
}

def normalize_label(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    if s.isdigit():
        return int(s)
    return label_map.get(s, None)

df["GT_norm"] = df[GT_COL].apply(normalize_label)

# =========================
# 4. 找模型列
# =========================
ignore_cols = {
    GT_COL,
    "GT_norm",
    "GPTImageType",
    "image_name",
    "image_file",
    "Image Name",
    "ID",
    "id"
}

model_cols = [c for c in df.columns if c not in ignore_cols]
print("模型列:", model_cols)

# =========================
# 5. 类别标签
# =========================
labels = [0, 1, 2, 3, 4]

# =========================
# 6. 结果存储
# =========================
summary_results = []
per_class_results = []

# =========================
# 7. 每个模型计算指标
# =========================
for col in model_cols:
    print(f"\n========== {col} ==========")

    df[f"{col}_norm"] = df[col].apply(normalize_label)

    valid_df = df[df["GT_norm"].notna() & df[f"{col}_norm"].notna()].copy()

    if len(valid_df) == 0:
        print(f"⚠️ {col} 没有可用数据，跳过")
        continue

    y_true = valid_df["GT_norm"].astype(int)
    y_pred = valid_df[f"{col}_norm"].astype(int)

    # -------- 整体指标 --------
    acc = accuracy_score(y_true, y_pred)

    prec_weighted = precision_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    rec_weighted = recall_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)

    prec_macro = precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    rec_macro = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    print(f"Accuracy           : {acc:.4f}")
    print(f"Precision_weighted : {prec_weighted:.4f}")
    print(f"Recall_weighted    : {rec_weighted:.4f}")
    print(f"F1_weighted        : {f1_weighted:.4f}")
    print(f"Precision_macro    : {prec_macro:.4f}")
    print(f"Recall_macro       : {rec_macro:.4f}")
    print(f"F1_macro           : {f1_macro:.4f}")
    print(f"Valid rows used    : {len(valid_df)}")

    # -------- 每类指标 --------
    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0
    )

    # confusion matrix 用来算每类“class accuracy”
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    print("\nPer-class metrics:")
    for idx, l in enumerate(labels):
        p = report[str(l)]["precision"]
        r = report[str(l)]["recall"]
        f = report[str(l)]["f1-score"]
        support = report[str(l)]["support"]

        # 每类 accuracy（one-vs-class hit rate 的直观写法）
        # 对该类来说，预测正确数 / 该类总数，其实和 recall 数值相同
        class_acc = cm[idx, idx] / cm[idx].sum() if cm[idx].sum() > 0 else 0

        print(f"Class {l}: Acc={class_acc:.3f} P={p:.3f} R={r:.3f} F1={f:.3f}")

        per_class_results.append({
            "model": col,
            "class": l,
            "class_accuracy": class_acc,
            "precision": p,
            "recall": r,
            "f1": f,
            "support": support
        })

    # -------- 保存整体 --------
    summary_results.append({
        "model": col,
        "valid_rows": len(valid_df),
        "accuracy": acc,
        "precision_weighted": prec_weighted,
        "recall_weighted": rec_weighted,
        "f1_weighted": f1_weighted,
        "precision_macro": prec_macro,
        "recall_macro": rec_macro,
        "f1_macro": f1_macro
    })

# =========================
# 8. 保存 CSV
# =========================
summary_df = pd.DataFrame(summary_results)
per_class_df = pd.DataFrame(per_class_results)

summary_df.to_csv(
    "/Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/LLM/Model_summary_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)
per_class_df.to_csv(
    "/Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/LLM/Model_per_class_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ 已输出:")
print(" - /Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/LLM/Model_summary_metrics.csv")
print(" - /Users/zixuanzhao/Desktop/MKG/JMI/JMI-New/LLM/Model_per_class_metrics.csv")

列名: ['image_name', 'GT_image_type', 'GT_output', 'Bytedance-seed-1.6', 'Gemini-2.5-Flash', 'Gemma-4-26b', 'GLM-4.6V', 'Grok-4.1-Fast', 'Qwen3.5-35b', 'Xiaomi-mimo-v2', 'Llama-3.2-11b-vision']
模型列: ['GT_image_type', 'Bytedance-seed-1.6', 'Gemini-2.5-Flash', 'Gemma-4-26b', 'GLM-4.6V', 'Grok-4.1-Fast', 'Qwen3.5-35b', 'Xiaomi-mimo-v2', 'Llama-3.2-11b-vision']

========== GT_image_type ==========
Accuracy           : 0.6685
Precision_weighted : 0.6712
Recall_weighted    : 0.6712
F1_weighted        : 0.6712
Precision_macro    : 0.4000
Recall_macro       : 0.4000
F1_macro           : 0.4000
Valid rows used    : 371

Per-class metrics:
Class 0: Acc=1.000 P=1.000 R=1.000 F1=1.000
Class 1: Acc=1.000 P=1.000 R=1.000 F1=1.000
Class 2: Acc=0.000 P=0.000 R=0.000 F1=0.000
Class 3: Acc=0.000 P=0.000 R=0.000 F1=0.000
Class 4: Acc=0.000 P=0.000 R=0.000 F1=0.000

========== Bytedance-seed-1.6 ==========
Accuracy           : 0.9272
Precision_weighted : 0.9428
Recall_weighted    : 0.9321
F1_weighted       